# Research Question: How does mutation count impact the overall survival in Melanoma patients, and does patient age correlate with higher mutational vulnerability?

## Stratified Tumor Mutational Burden (TMB) & Survival Analysis in TCGA Melanoma

## Project Overview
My project investigates the biological interplay between somatic driver mutations (*BRAF*, *NRAS*), nonsynonymous Tumor Mutational Burden (TMB), and overall survival in Skin Cutaneous Melanoma using TCGA Firehose Legacy data via cBioPortal. The analysis progresses from exploratory correlation checks to driver-stratified distribution profiling and non-parametric Kaplan-Meier survival modeling.

---

## Key Results & Biological Insights

### 1. Exploratory Analyses: Demographics & Mutational Load
* **Age vs. TMB Correlation:** Evaluated continuous patient age against nonsynonymous TMB. A weak positive trend was observed with r = 0.1, p = 0.067. While consistent with somatic mutation accumulation over time, it did not cross the strict threshold for statistical significance since 0.067 > 0.05

### 2. Driver Subtype Stratification (*BRAF* vs. *NRAS* vs. Wild-Type)
* **Central Tendency:** *NRAS*-mutant tumors displayed the highest median TMB (~ 13 mutations/Mb), followed by *BRAF*-mutants (~ 9 mutations/Mb), while Wild-Type tumors showed a markedly lower median (~ 2-3 mutations/Mb).
* **Dispersion & Skewness:** The Wild-Type cohort exhibited a strong right-skew and the largest Interquartile Range, which reflects underlying molecular heterogeneity across non-canonical drivers (some examples of them being *NF1*, *KIT*, and triple-wild-type). Conversely, *BRAF* and *NRAS*  displayed tighter, more symmetrical interquartile ranges.
* **Overlap:** There was a substantial vertical overlap across all three groups. *NRAS* mutants have ~ 35% of their box elevated above *BRAF*, but driver status alone is insufficient to predict total mutational burden.
* **Methodological Visualization:** The y-axis was truncated at 80 mutations/Mb to resolve medians and IQRs visibility issues. Hypermutated samples  (> 80 TMB, > max 1000 TMB) were retained in all statistical testing but excluded from axis scaling to prevent compression of the data.
* **Biological Mechanism:** The elevated TMB in both the *BRAF* and *NRAS* groups aligns with MAPK pathway hyperactivation, which frequently occurs with high baseline ultraviolet radiation damage signatures.

### 3. Kaplan-Meier Survival Analysis (High vs. Low TMB)
* **Early & Mid-Term Divergence (0–100 Months):** Survival curves for High vs. Low TMB groups separate early at around 10-20 months and maintain consistent separation. High TMB patients demonstrate a clear overall survival advantage during this window.
* **Statistical Significance:** A Log-Rank test confirmed the survival difference is statistically significant (p = 0.0018). This outperformed preliminary binary $t$-tests (p = 0.0828) by correctly accounting for right-censored clinical tracking, which the preliminary $t$-tests did not account for.
* **Tail Instability (100–350 Months):** Across the extended 350-month follow-up, 95% Confidence Intervals widen significantly past 150 months due to heavy right-censoring as fewer patients remain at risk.
* **Immunological Basis:** UV-induced somatic mutations generate tumor-specific neoantigens presented on the host MHC molecules. Higher TMB increases neoantigen diversity. which enhances T-cell recognition, endogenous anti-tumor immune surveillance, and overall survival.

### 4. Predictive Machine Learning & Clinical Decision Support
* **Random Survival Forests (scikit-survival):** I trained an ensemble survival model on patient age, nonsynonymous TMB, and driver mutation status (*BRAF* and *NRAS*). I achieved a training C-Index of 0.7067 and a testing C-Index of 0.6280 on held-out clinical data.
* **Censoring Handling & Validation:** Unlike standard classification algorithms, the RSF explicitly handles right-censored longitudinal outcomes (T > 0 strictly enforced) out to 350 months using log-rank split criteria.
* **Permutation Feature Importance:** I quantified feature contributions across test-set predictions by establishing the relative predictive weights of continuous TMB, patient age, and driver subtypes.
* **Interactive Clinical Simulator (ipywidgets):** I built a real-time, in-notebook decision support tool that takes user-defined patient parameters such as age, TMB, and Driver Subtype, and dynamically plots predicted individual survival trajectories.

---

## Technical Stack & Dependencies
* **Data Processing & Analytics:** Python, pandas, numpy, scipy
* **Biostatistics & Survival Modeling:** Lifelines (Kaplan-Meier Fitter, Log-Rank Test), scikit-survival (Random Survival Forest)
* **Data Visualization:** seaborn, matplotlib, ipywidgets
* **Data Source:** cBioPortal: TCGA Skin Cutaneous Melanoma (Firehose Legacy)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# 1. Load patient and sample clinical datasets
df_patient = pd.read_csv('data_clinical_patient.txt', sep='\t', skiprows=4)
df_sample = pd.read_csv('data_clinical_sample.txt', sep='\t', skiprows=4)

# 2. Merge datasets on PATIENT_ID
df_merged = pd.merge(df_patient, df_sample, on='PATIENT_ID')

# 3. Select target columns
df_subset = df_merged[['PATIENT_ID', 'AGE', 'OS_STATUS', 'OS_MONTHS', 'TMB_NONSYNONYMOUS']].copy()

# 4. Convert to numeric, forcing non-numeric strings ('[Not Available]') to NaN
df_subset['AGE'] = pd.to_numeric(df_subset['AGE'], errors='coerce')
df_subset['OS_MONTHS'] = pd.to_numeric(df_subset['OS_MONTHS'], errors='coerce')
df_subset['TMB_NONSYNONYMOUS'] = pd.to_numeric(df_subset['TMB_NONSYNONYMOUS'], errors='coerce')

# 5. Drop rows with NaN in any of our analysis columns
df_clean = df_subset.dropna().copy()

print(f"Successfully cleaned data! {len(df_clean)} patients ready for analysis.")
df_clean.head()

In [ ]:
# 1. Setup 1x2 visual panel
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 2. Plot 1: Age vs. Tumor Mutational Burden
sns.scatterplot(ax=axes[0], data=df_clean, x='AGE', y='TMB_NONSYNONYMOUS', alpha=0.6, color='teal')
axes[0].set_title('Age vs. Tumor Mutational Burden (TMB)')
axes[0].set_xlabel('Age at Diagnosis')
axes[0].set_ylabel('Nonsynonymous TMB')

# Compute Pearson correlation
r, p_val = stats.pearsonr(df_clean['AGE'], df_clean['TMB_NONSYNONYMOUS'])
axes[0].annotate(f'r = {r:.2f}\np = {p_val:.3f}', xy=(0.05, 0.85), xycoords='axes fraction', 
                 bbox=dict(boxstyle="round", fc="white"))

# 3. Plot 2: Survival Months by High vs. Low TMB Group (Warning Fix Applied)
median_tmb = df_clean['TMB_NONSYNONYMOUS'].median()
df_clean['TMB_Group'] = df_clean['TMB_NONSYNONYMOUS'].apply(lambda x: 'High TMB' if x >= median_tmb else 'Low TMB')

sns.boxplot(
    ax=axes[1], 
    data=df_clean, 
    x='TMB_Group', 
    y='OS_MONTHS', 
    hue='TMB_Group', 
    legend=False, 
    palette='Set2'
)
axes[1].set_title('Overall Survival Months by TMB Group')
axes[1].set_xlabel('TMB Classification (Median Split)')
axes[1].set_ylabel('Survival (Months)')

plt.tight_layout()
plt.show()

# 4. Run Two-Sample T-test
high_tmb = df_clean[df_clean['TMB_Group'] == 'High TMB']['OS_MONTHS']
low_tmb = df_clean[df_clean['TMB_Group'] == 'Low TMB']['OS_MONTHS']
t_stat, t_p_val = stats.ttest_ind(high_tmb, low_tmb)

print(f"T-Test Results (High vs Low TMB Survival): t = {t_stat:.2f}, p-value = {t_p_val:.4f}")

In [ ]:
# Map OS_STATUS string to binary numeric column (1 = Deceased, 0 = Living/Censored)
df_clean['DECEASED'] = df_clean['OS_STATUS'].apply(
    lambda status: 1 if 'DECEASED' in str(status).upper() else 0
)

# Check the event vs censored breakdown
print(df_clean['DECEASED'].value_counts())

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test

# 1st step is to parse through mutation data for driver genes

df_mut = pd.read_csv('data_mutations.txt', sep='\t', comment='#', low_memory=False)
df_mut.columns = df_mut.columns.str.strip()

braf_pts = set(df_mut[df_mut['Hugo_Symbol'] == 'BRAF']['Tumor_Sample_Barcode'].str[:12])
nras_pts = set(df_mut[df_mut['Hugo_Symbol'] == 'NRAS']['Tumor_Sample_Barcode'].str[:12])

def categorize_driver(pid):
    if pid in braf_pts:
        return 'BRAF Mutant'
    elif pid in nras_pts:
        return 'NRAS Mutant'
    return 'Wild-Type'

df_clean['Driver_Status'] = df_clean['PATIENT_ID'].apply(categorize_driver)

# 2nd step is to plot the driver subtype versus TMB

plt.figure(figsize=(8, 5))
sns.boxplot(data=df_clean, x='Driver_Status', y='TMB_NONSYNONYMOUS', hue='Driver_Status', palette='Set1', legend=False, showfliers = False)
plt.ylim(0, 80)
plt.title('Tumor Mutational Burden by Melanoma Driver Subtype')
plt.ylabel('Nonsynonymous TMB')
plt.xlabel('Driver Gene Mutation Status')
plt.tight_layout()
plt.show()

# 3rd step is to show the true survival analysis through the Kaplan-Meier Curves

kmf = KaplanMeierFitter()
fig, ax = plt.subplots(figsize=(9, 6))

# 4th step is to plot high vs low TMB groups while taking into account censoring 

high_tmb = df_clean[df_clean['TMB_Group'] == 'High TMB']
low_tmb = df_clean[df_clean['TMB_Group'] == 'Low TMB']

kmf.fit(high_tmb['OS_MONTHS'], event_observed=high_tmb['DECEASED'], label='High TMB')
kmf.plot_survival_function(ax=ax, ci_show=True)

kmf.fit(low_tmb['OS_MONTHS'], event_observed=low_tmb['DECEASED'], label='Low TMB')
kmf.plot_survival_function(ax=ax, ci_show=True)

plt.title('Kaplan-Meier Survival Estimates (TCGA Melanoma Cohort)')
plt.xlabel('Survival Time (Months)')
plt.ylabel('Probability of Survival')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

# Final step is to run the Log-Rank test

lr_results = logrank_test(
    high_tmb['OS_MONTHS'], low_tmb['OS_MONTHS'],
    event_observed_A=high_tmb['DECEASED'], event_observed_B=low_tmb['DECEASED']
)
print(f"Log-Rank Test p-value: {lr_results.p_value:.4f}")


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sksurv.ensemble import RandomSurvivalForest

source_df = df_clean 

# 1. Select features, drop illegitmate ones, and strictly filter for OS_MONTHS > 0
feature_cols = ['AGE', 'TMB_NONSYNONYMOUS', 'Driver_Status']
df_ml = source_df[['OS_MONTHS', 'DECEASED'] + feature_cols].dropna().copy()
df_ml = df_ml[df_ml['OS_MONTHS'] > 0].copy()  # Prevents sksurv time <= 0 error

# 2. One-Hot Encode categorical driver status (BRAF, NRAS, Wild-Type)
X = pd.get_dummies(df_ml[feature_cols], columns=['Driver_Status'], drop_first=True)

# 3. Create structured array target (y) for sksurv
y = np.array(
    list(zip(df_ml['DECEASED'].astype(bool), df_ml['OS_MONTHS'].astype(float))),
    dtype=[('Status', '?'), ('Time', '<f8')]
)

# 4. Train/Test Split (80/20) stratified by event status
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=[e[0] for e in y]
)

# 5. Fit Random Survival Forest
rsf = RandomSurvivalForest(
    n_estimators=100,
    min_samples_split=10,
    min_samples_leaf=15,
    max_features="sqrt",
    n_jobs=-1,
    random_state=42
)

rsf.fit(X_train, y_train)

# 6. Evaluate C-Index
print(f"Training C-Index: {rsf.score(X_train, y_train):.4f}")
print(f"Testing C-Index:  {rsf.score(X_test, y_test):.4f}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.inspection import permutation_importance

# Calculate feature importances using sklearn's inspection module
result = permutation_importance(rsf, X_test, y_test, n_repeats=10, random_state=42)

# Format into DataFrame
importance_df = pd.DataFrame(
    {
        'Feature': X_test.columns,
        'Importance_Mean': result.importances_mean,
        'Importance_Std': result.importances_std
    }
).sort_values(by='Importance_Mean', ascending=False)

print("--- Feature Importance Ranking ---")
print(importance_df.to_string(index=False))

# Plot Feature Importances
plt.figure(figsize=(8, 4))
plt.barh(importance_df['Feature'], importance_df['Importance_Mean'], color='teal')
plt.xlabel("Mean Decrease in C-Index (Permutation Importance)")
plt.title("Random Survival Forest: Feature Importance Ranking")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig("rsf_feature_importance.png", dpi=300)
plt.show()

In [ ]:
# Predict individual survival functions for representative test patients
surv_funcs = rsf.predict_survival_function(X_test.iloc[:3])

plt.figure(figsize=(9, 5))
for i, surv_fn in enumerate(surv_funcs):
    plt.step(surv_fn.x, surv_fn.y, where="post", label=f"Test Patient {i+1}")

plt.ylabel("Predicted Survival Probability")
plt.xlabel("Time in Months")
plt.title("RSF Predicted Individual Patient Survival Curves")
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()
plt.tight_layout()
plt.savefig("rsf_individual_curves.png", dpi=300)
plt.show()

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd
import matplotlib.pyplot as plt

# 1. Build Interactive Input UI
age_slider = widgets.IntSlider(value=55, min=18, max=90, step=1, description='Age:')
tmb_slider = widgets.FloatSlider(value=10.0, min=0.1, max=80.0, step=0.5, description='TMB (mut/Mb):')
driver_dropdown = widgets.Dropdown(
    options=['Wild-Type', 'BRAF-mutant', 'NRAS-mutant'],
    value='BRAF-mutant',
    description='Driver Status:'
)
predict_button = widgets.Button(description='Predict Survival Curve', button_style='primary', icon='chart-line')
output = widgets.Output()

# 2. Define Prediction Callback Function
def generate_patient_prediction(b):
    with output:
        clear_output(wait=True)
        
        # Base input frame
        patient_data = pd.DataFrame([{
            'AGE': age_slider.value,
            'TMB_NONSYNONYMOUS': tmb_slider.value
        }])
        
        # Reconstruct dummy/one-hot encoding to match training matrix X
        for col in X.columns:
            if col not in patient_data.columns:
                # Check matching categorical column name
                if f"Driver_Status_{driver_dropdown.value}" in col:
                    patient_data[col] = 1
                elif "Driver" in col and driver_dropdown.value != 'Wild-Type':
                    # Fallback for generic binary indicator
                    patient_data[col] = 1 if driver_dropdown.value in col else 0
                else:
                    patient_data[col] = 0

        # Enforce exact feature order from training matrix
        patient_data = patient_data[X.columns] 

        # Predict survival function using trained RSF
        surv_fn = rsf.predict_survival_function(patient_data)[0]

        # Render Dynamic Curve
        fig, ax = plt.subplots(figsize=(9, 5))
        ax.step(surv_fn.x, surv_fn.y, where="post", color="darkred", linewidth=2.5)
        ax.set_title(f"Predicted Overall Survival Trajectory\n(Age: {age_slider.value} | TMB: {tmb_slider.value} mut/Mb | Subtype: {driver_dropdown.value})")
        ax.set_xlabel("Follow-Up Time (Months)")
        ax.set_ylabel("Predicted Survival Probability")
        ax.set_ylim(0, 1.05)
        ax.grid(True, linestyle="--", alpha=0.5)
        plt.tight_layout()
        plt.show()
        plt.close(fig) # Flush plot buffer

predict_button.on_click(generate_patient_prediction)

# 3. Display UI Controls
display(widgets.VBox([
    widgets.HTML("<h3>TCGA Melanoma Patient Survival Simulator</h3>"),
    age_slider,
    tmb_slider,
    driver_dropdown,
    predict_button,
    output
]))

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

def run_interactive_survival_predictor(save_plot=True):
    """
    Console-based patient survival predictor with clinical input validation.
    Enforces realistic biological bounds (Age: 18-100, TMB: 0.1-200 mut/Mb).
    """
    print("=" * 60)
    print("  TCGA MELANOMA PATIENT SURVIVAL PREDICTOR (RSF MODEL)")
    print("=" * 60)
    
    # 1. Collect & validate Age input (Bounds: 18 to 100)
    try:
        raw_age = float(input("Enter Patient Age (18-100, Default: 55): ") or 55)
        age_val = max(18.0, min(100.0, raw_age))  # Clamp value between 18 and 100
        if raw_age != age_val:
            print(f" Age adjusted to valid clinical range: {age_val}")
    except ValueError:
        age_val = 55.0
        print(" Invalid age input. Defaulting to 55.")

    # 2. Collect & validate TMB input (Bounds: 0.1 to 200.0 mut/Mb)
    try:
        raw_tmb = float(input("Enter TMB in mut/Mb (0.1-200.0, Default: 10.0): ") or 10.0)
        tmb_val = max(0.1, min(200.0, raw_tmb))  # Clamp value between 0.1 and 200.0
        if raw_tmb != tmb_val:
            print(f" TMB adjusted to valid clinical range: {tmb_val} mut/Mb")
    except ValueError:
        tmb_val = 10.0
        print(" Invalid TMB input. Defaulting to 10.0 mut/Mb.")

    # 3. Collect Driver Subtype
    print("\nDriver Subtypes:")
    print(" [1] BRAF-mutant")
    print(" [2] NRAS-mutant")
    print(" [3] Wild-Type")
    subtype_choice = input("Select Driver Subtype [1-3] (Default: 1): ").strip()
    
    subtype_map = {'1': 'BRAF-mutant', '2': 'NRAS-mutant', '3': 'Wild-Type'}
    driver_val = subtype_map.get(subtype_choice, 'BRAF-mutant')

    # 4. Format inputs into DataFrame matching matrix X
    patient_data = pd.DataFrame([{
        'AGE': age_val,
        'TMB_NONSYNONYMOUS': tmb_val
    }])
    
    for col in X.columns:
        if col not in patient_data.columns:
            if driver_val != 'Wild-Type' and driver_val in col:
                patient_data[col] = 1
            else:
                patient_data[col] = 0
                
    patient_data = patient_data[X.columns]
    
    # 5. Predict survival trajectory using trained RSF
    surv_fn = rsf.predict_survival_function(patient_data)[0]
    
    # 6. Render Matplotlib Plot
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.step(surv_fn.x, surv_fn.y, where="post", color="darkred", linewidth=2.5)
    ax.set_title(
        f"Predicted Overall Survival Trajectory\n"
        f"(Age: {int(age_val)} | TMB: {tmb_val} mut/Mb | Subtype: {driver_val})", 
        fontsize=12, 
        fontweight='bold'
    )
    ax.set_xlabel("Follow-Up Time (Months)", fontsize=10)
    ax.set_ylabel("Predicted Survival Probability", fontsize=10)
    ax.set_ylim(0, 1.05)
    ax.grid(True, linestyle="--", alpha=0.5)
    plt.tight_layout()
    
    # 7. Export directly to figures directory
    if save_plot:
        os.makedirs('figures', exist_ok=True)
        export_path = 'figures/rsf_predicted_survival_curves.png'
        plt.savefig(export_path, dpi=300, bbox_inches='tight')
        print(f"\n Success! Plot saved to: {export_path}")
        
    plt.show()

# Run predictor
run_interactive_survival_predictor()